# Cantilever: Solid–Frame Coupling visualised in pyGmshViewer

Same model as `cantilever_solid_frame_coupling.ipynb` (a shell-element
“solid” block coupled to an `elasticBeamColumn` frame via a master node,
duplicated interface nodes, `rigidLink('beam')` and `equalDOF`), but now we
export the result to VTU and load it in **pyGmshViewer** so the coupling
region can be inspected interactively.

Three meshes are loaded side-by-side in the viewer:

| Mesh             | Cells                | Displays                          |
|------------------|----------------------|-----------------------------------|
| `solid`          | `VTK_QUAD`           | Shell region, deformed shape      |
| `frame`          | `VTK_LINE`           | Frame element (master→tip)        |
| `rigid_links`    | `VTK_LINE`           | Dashed links master→interface     |

All three share the same `Displacement` vector field, so toggling
**Show Deformed** animates the coupling: the frame rotates, the rigid
links move like spokes of a wheel, and the duplicated–solid interface
nodes follow along.

## 1. Parameters

In [1]:
import openseespy.opensees as ops
import numpy as np
import matplotlib.pyplot as plt

# Geometry
L      = 10.0        # Total beam length [m]
h      = 1.0         # Beam depth [m]
b      = 0.5         # Beam width (out-of-plane thickness) [m]

# Material
E      = 30.0e6      # Young's modulus [kPa]
nu     = 0.2         # Poisson's ratio
G      = E / (2.0 * (1.0 + nu))

# Loading
P      = -100.0      # Tip point load [kN] (negative = downward)

# Domain split
L_solid = L / 2.0
L_frame = L / 2.0

# Solid mesh density
nx_s = 20            # elements along x
ny_s = 8             # elements through depth

dx = L_solid / nx_s
dy = h / ny_s

# Frame cross-section (rectangular b x h)
A  = b * h
Iz = b * h**3 / 12.0    # in-plane bending
Iy = h * b**3 / 12.0    # out-of-plane bending
J  = 0.5 * (Iy + Iz)    # torsion (approx)

print(f"Beam: L={L} m, h={h} m, b={b} m")
print(f"Solid region: {nx_s}x{ny_s} elements, dx={dx:.3f}, dy={dy:.4f}")
print(f"Section: A={A}, Iz={Iz:.6f}, EI={E*Iz:.2e}")

Beam: L=10.0 m, h=1.0 m, b=0.5 m
Solid region: 20x8 elements, dx=0.250, dy=0.1250
Section: A=0.5, Iz=0.041667, EI=1.25e+06


## 2. Build the Model

We use `ndm=3, ndf=6` so both shell and beam-column elements are compatible.

In [2]:
ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

### 2.1 Solid Region Nodes

Grid of $(n_{x}+1) \times (n_{y}+1)$ nodes in the XY plane at $z=0$. All nodes are constrained to 2D by fixing $u_z$, $\theta_x$, and $\theta_y$.

In [3]:
solid_nodes = {}   # (i, j) -> node_tag
nid = 0

for j in range(ny_s + 1):
    for i in range(nx_s + 1):
        nid += 1
        x = i * dx
        y = -h / 2.0 + j * dy
        ops.node(nid, x, y, 0.0)
        solid_nodes[(i, j)] = nid

n_solid_nodes = nid

# Identify interface nodes (right edge of solid, i = nx_s)
interface_solid_tags = [solid_nodes[(nx_s, j)] for j in range(ny_s + 1)]

# Constrain to 2D: fix uz(3), rx(4), ry(5)
for (i, j), tag in solid_nodes.items():
    if i == 0:
        continue   # left edge gets full fixity below
    ops.fix(tag, 0, 0, 1, 1, 1, 0)  # free: ux, uy, rz

# Fixed support at x=0
for j in range(ny_s + 1):
    ops.fix(solid_nodes[(0, j)], 1, 1, 1, 1, 1, 1)

print(f"Solid nodes: {n_solid_nodes}")
print(f"Interface nodes: {interface_solid_tags}")

Solid nodes: 189
Interface nodes: [21, 42, 63, 84, 105, 126, 147, 168, 189]


### 2.2 Frame Nodes

In [4]:
master_tag = n_solid_nodes + 1   # at cross-section centroid of interface
tip_tag    = n_solid_nodes + 2   # free end

ops.node(master_tag, L_solid, 0.0, 0.0)
ops.node(tip_tag,    L,       0.0, 0.0)

# Constrain to 2D
ops.fix(master_tag, 0, 0, 1, 1, 1, 0)
ops.fix(tip_tag,    0, 0, 1, 1, 1, 0)

print(f"Master node: {master_tag} at ({L_solid}, 0)")
print(f"Tip node:    {tip_tag} at ({L}, 0)")

Master node: 190 at (5.0, 0)
Tip node:    191 at (10.0, 0)


### 2.3 Duplicated Interface Nodes

These are the key to the coupling. Each duplicated node:
- Is **co-located** with a solid interface node
- Has **full 6 DOFs** (can receive rotation from the rigid link)
- Is **not connected** to any element directly — it exists purely as a constraint intermediary
- Has **no fixity** applied — all its DOFs are controlled by the `rigidLink`

In [5]:
dup_tags = []

for j in range(ny_s + 1):
    dn = tip_tag + 1 + j
    y  = -h / 2.0 + j * dy
    ops.node(dn, L_solid, y, 0.0)
    # NO fixity — rigidLink constrains all 6 DOFs from the master
    dup_tags.append(dn)

print(f"Duplicated nodes: {dup_tags[0]}..{dup_tags[-1]} ({len(dup_tags)} nodes)")
print(f"Total model nodes: {dup_tags[-1]}")

Duplicated nodes: 192..200 (9 nodes)
Total model nodes: 200


## 3. Elements

### 3.1 Shell Elements (solid region)

We use `ShellMITC4` with an `ElasticMembranePlateSection`. The membrane component provides in-plane (solid-like) behavior, and the plate component provides bending. The thickness parameter is the beam width $b$.

In [6]:
# Shell section
ops.section('ElasticMembranePlateSection', 1, E, nu, b, 0.0)

# Build shell elements
eid = 0
quad_elements = []

for j in range(ny_s):
    for i in range(nx_s):
        eid += 1
        n1 = solid_nodes[(i,   j)]
        n2 = solid_nodes[(i+1, j)]
        n3 = solid_nodes[(i+1, j+1)]
        n4 = solid_nodes[(i,   j+1)]
        ops.element('ShellMITC4', eid, n1, n2, n3, n4, 1)
        quad_elements.append((eid, n1, n2, n3, n4))

n_solid_ele = eid
print(f"Shell elements: {n_solid_ele}")

Shell elements: 160


### 3.2 Frame Element

3D `elasticBeamColumn` with `vecxz = (0, 0, 1)` so that:
- Local z $\approx$ global Z → $I_z$ controls out-of-plane bending (not relevant here)
- Local y $\approx$ global Y → $I_y$ controls in-plane bending... 

Actually, for the `elasticBeamColumn` 3D syntax: `(tag, ni, nj, A, E, G, J, Iy, Iz, transf)`
- $I_z$: bending about local z → curvature in local xy plane → **in-plane bending** (our bending)
- $I_y$: bending about local y → curvature in local xz plane → out-of-plane bending

In [7]:
ops.geomTransf('Linear', 1, 0.0, 0.0, 1.0)

eid += 1
frame_eid = eid
ops.element('elasticBeamColumn', eid, master_tag, tip_tag,
            A, E, G, J, Iy, Iz, 1)

print(f"Frame element: {frame_eid}")
print(f"Total elements: {eid}")

Frame element: 161
Total elements: 161


## 4. Coupling Constraints

This is the core of the example. Two steps:

### Step A: Rigid Links (master → duplicated nodes)

`rigidLink('beam', master, dupNode)` constrains all 6 DOFs of each dup node based on rigid body kinematics:

$$u^{\text{dup}}_x = u^{\text{master}}_x - (y_{\text{dup}} - y_{\text{master}}) \cdot \theta^{\text{master}}_z$$
$$u^{\text{dup}}_y = u^{\text{master}}_y$$
$$\theta^{\text{dup}}_z = \theta^{\text{master}}_z$$

(The $x$-distance term vanishes because all interface nodes share the same $x$-coordinate.)

This enforces **plane sections remain plane** at the interface.

### Step B: Equal DOF (duplicated → solid nodes)

`equalDOF(dupNode, solidNode, 1, 2)` ties only the **translational** DOFs:

$$u^{\text{solid}}_x = u^{\text{dup}}_x, \quad u^{\text{solid}}_y = u^{\text{dup}}_y$$

The rotational DOF $\theta_z$ is **not** transferred to the solid node — the solid element has no rotational stiffness and should not receive a rotation constraint.

### Constraint Handler

The dup nodes are **slaves** in the `rigidLink` MP constraint and **retained** (masters) in the `equalDOF` MP constraint. This creates **nested multi-point constraints**. The `Transformation` handler cannot handle this nesting, so we use the **Penalty** handler.

In [8]:
# Step A: rigidLink — master to each duplicated node
for dn in dup_tags:
    ops.rigidLink('beam', master_tag, dn)

# Step B: equalDOF — dup (retained) to solid (constrained), DOFs 1,2 only
for dn, sn in zip(dup_tags, interface_solid_tags):
    ops.equalDOF(dn, sn, 1, 2)

print(f"Rigid links: {len(dup_tags)} (master -> dup nodes)")
print(f"EqualDOF:    {len(dup_tags)} (dup -> solid, DOFs 1,2)")

Rigid links: 9 (master -> dup nodes)
EqualDOF:    9 (dup -> solid, DOFs 1,2)


## 5. Loading & Analysis

In [9]:
# Point load at tip
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)
ops.load(tip_tag, 0.0, P, 0.0, 0.0, 0.0, 0.0)

# Analysis (Penalty for nested MP constraints)
ops.constraints('Penalty', 1.0e14, 1.0e14)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 50)
ops.algorithm('Newton')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

print("Running analysis (10 load steps)...")
ok = ops.analyze(10)

if ok != 0:
    print("*** Analysis FAILED ***")
else:
    print("Analysis completed successfully!")

Running analysis (10 load steps)...
Analysis completed successfully!


## 6. Results

### 6.1 Tip and Master Displacements

In [10]:
# DOF mapping: ux=1, uy=2, uz=3, rx=4, ry=5, rz=6
tip_d    = ops.nodeDisp(tip_tag)
master_d = ops.nodeDisp(master_tag)

print(f"Tip    (x={L:5.1f}): ux={tip_d[0]:+.6e}  uy={tip_d[1]:+.6e}  rz={tip_d[5]:+.6e}")
print(f"Master (x={L_solid:5.1f}): ux={master_d[0]:+.6e}  uy={master_d[1]:+.6e}  rz={master_d[5]:+.6e}")

# Analytical reference
delta_EB    = P * L**3 / (3.0 * E * Iz)
delta_shear = P * L / (5.0/6.0 * A * G)
delta_Timo  = delta_EB + delta_shear

print(f"\nEuler-Bernoulli:  {delta_EB:+.6e} m")
print(f"Timoshenko:       {delta_Timo:+.6e} m")
print(f"Numerical:        {tip_d[1]:+.6e} m")
print(f"\nRatio (num/EB):         {tip_d[1]/delta_EB:.4f}")
print(f"Ratio (num/Timoshenko): {tip_d[1]/delta_Timo:.4f}")

Tip    (x= 10.0): ux=-3.782195e-18  uy=-2.596588e-02  rz=-3.898953e-03
Master (x=  5.0): ux=-3.782195e-18  uy=-8.137786e-03  rz=-2.898953e-03

Euler-Bernoulli:  -2.666667e-02 m
Timoshenko:       -2.685867e-02 m
Numerical:        -2.596588e-02 m

Ratio (num/EB):         0.9737
Ratio (num/Timoshenko): 0.9668


---
## Viewer export — Export the coupling to VTU

We split the model into three VTU files so each can carry its own cell
type:

- `coupling_solid.vtu`      — the shell region as `VTK_QUAD` cells
- `coupling_frame.vtu`      — the frame element as a single `VTK_LINE`
- `coupling_rigid_links.vtu` — the rigid links as `VTK_LINE` cells

All three share a `Displacement` point field so the viewer can deform
them consistently.

In [ ]:
# ── Build arrays for the viewer ──
# 1) Solid mesh (quads)
solid_tag_list = sorted(solid_nodes.values())
tag_to_row = {t: i for i, t in enumerate(solid_tag_list)}

solid_points = np.zeros((len(solid_tag_list), 3))
solid_disp   = np.zeros((len(solid_tag_list), 3))
for (i, j), tag in solid_nodes.items():
    row = tag_to_row[tag]
    solid_points[row] = (i * dx, -h/2.0 + j * dy, 0.0)
    d = ops.nodeDisp(tag)
    solid_disp[row] = (d[0], d[1], d[2])

solid_cells = np.array([
    [tag_to_row[solid_nodes[(i,   j)]],
     tag_to_row[solid_nodes[(i+1, j)]],
     tag_to_row[solid_nodes[(i+1, j+1)]],
     tag_to_row[solid_nodes[(i,   j+1)]]]
    for j in range(ny_s) for i in range(nx_s)
], dtype=np.int32)

print(f'Solid mesh : {len(solid_tag_list)} nodes, {len(solid_cells)} quads')
print(f'  |u| max  : {np.linalg.norm(solid_disp, axis=1).max():.4e} m')

In [ ]:
# 2) Frame element (line from master to tip)
master_xyz = np.array(ops.nodeCoord(master_tag)); master_d = np.array(ops.nodeDisp(master_tag))[:3]
tip_xyz    = np.array(ops.nodeCoord(tip_tag));    tip_d    = np.array(ops.nodeDisp(tip_tag))[:3]

frame_points = np.vstack([master_xyz, tip_xyz])
frame_disp   = np.vstack([master_d, tip_d])
frame_cells  = np.array([[0, 1]], dtype=np.int32)

print(f'Frame      : master={master_xyz}  tip={tip_xyz}')
print(f'             tip disp = ({tip_d[0]:.4e}, {tip_d[1]:.4e}, {tip_d[2]:.4e}) m')

In [ ]:
# 3) Rigid links (master → each duplicated interface node)
# Each link = one line cell. Duplicated nodes share position with the
# solid interface nodes and move identically (rigidLink kinematics).
n_dup = len(dup_tags)

links_points = np.zeros((1 + n_dup, 3))
links_disp   = np.zeros((1 + n_dup, 3))

# Node 0 = master
links_points[0] = master_xyz
links_disp[0]   = master_d

for k, dn in enumerate(dup_tags):
    c = ops.nodeCoord(dn)
    d = ops.nodeDisp(dn)
    links_points[k+1] = c
    links_disp[k+1]   = d[:3]

# One line cell per duplicated node: master → dup_k
links_cells = np.array([[0, k+1] for k in range(n_dup)], dtype=np.int32)
print(f'Rigid links: {n_dup} lines (master → dup_0..dup_{n_dup-1})')

In [ ]:
# ── Write three VTU files ──
import sys
from pathlib import Path
sys.path.insert(0, '../src')
from pyGmsh.VTKExport import write_vtu

VTK_QUAD = 9
VTK_LINE = 3

solid_vtu = Path('coupling_solid.vtu').resolve()
frame_vtu = Path('coupling_frame.vtu').resolve()
links_vtu = Path('coupling_rigid_links.vtu').resolve()

write_vtu(
    solid_vtu, points=solid_points, cells=solid_cells,
    vtk_cell_type=VTK_QUAD,
    point_data={
        'Displacement': solid_disp,
        '|U|':          np.linalg.norm(solid_disp, axis=1),
        'Ux':           solid_disp[:, 0],
        'Uy':           solid_disp[:, 1],
    },
)
write_vtu(
    frame_vtu, points=frame_points, cells=frame_cells,
    vtk_cell_type=VTK_LINE,
    point_data={
        'Displacement': frame_disp,
        '|U|':          np.linalg.norm(frame_disp, axis=1),
    },
)
write_vtu(
    links_vtu, points=links_points, cells=links_cells,
    vtk_cell_type=VTK_LINE,
    point_data={
        'Displacement': links_disp,
        '|U|':          np.linalg.norm(links_disp, axis=1),
    },
)

print('Wrote:')
print(' ', solid_vtu.name)
print(' ', frame_vtu.name)
print(' ', links_vtu.name)

In [ ]:
# ── Launch pyGmshViewer with all three meshes loaded ──
# Install once: pip install -e ".[viewer]"  from the repo root
from pyGmshViewer import show

# show() takes variadic filepaths — pass each one individually
show(solid_vtu, frame_vtu, links_vtu, blocking=False)

print('Viewer launched. To see the coupling interaction:')
print('  1. In the Model tree, select each of the three loaded meshes')
print('     (coupling_solid, coupling_frame, coupling_rigid_links)')
print('  2. Controls → Contour: pick |U| (or Uy for the tip deflection)')
print('  3. Controls → Deformation: "Show Deformed", scale 50x or 100x')
print('  4. Switch coupling_rigid_links to Wireframe so the spokes')
print('     radiating from the master node are visually clean')
print('  5. Tools → Point Probe (Ctrl+P): click a dup-node on the rigid')
print('     link ends and confirm its Displacement matches the solid')
print('     interface node at the same location (equalDOF kinematics)')